# Person 1 — Perishable Goods Spoilage Classifier
## Version A: Advanced Features (Sensor + Logistics)

> **Role:** Person 1 — Tech Lead & ML Classifier (Step 7: FEFO & Warehouse Stock Management)  
> **Dataset:** `data/raw/7. perishable_goods_management.csv` (100,000 rows, 42 columns)  
> **Target:** `risk_label` — multiclass **`HIGH / MODERATE / LOW`**  
> **Output contract:** `{"label": "HIGH"|"MODERATE"|"LOW", "score": float}` ← same as `src/ml/spoilage.py`

---

### Target Label Derivation Logic
The column `was_spoiled` (did the product factually spoil?) is combined with `days_until_expiry` to build a 3-class label that matches the production output contract:

```
was_spoiled == 1               → HIGH     (factually spoiled — worst case)
was_spoiled == 0, DTE ≤  7    → HIGH     (< 1 week left, at high risk)
was_spoiled == 0, 7 < DTE ≤ 30 → MODERATE (needs attention)
was_spoiled == 0, DTE > 30    → LOW      (safe)
```

> `was_spoiled` and `days_until_expiry` are used **only to build the label**. `was_spoiled` is then **dropped from features** (it is a future outcome — keeping it as a feature would be leakage).

---

### Anti-Leakage Contract
| Column | Why dropped |
|---|---|
| `was_spoiled` | Used to derive label, then dropped — future outcome |
| `units_wasted` | Counted after spoilage occurs |
| `waste_pct` | Derived from `units_wasted` |
| `waste_cost` | Financial loss computed post-spoilage |
| `units_sold` | Known only after transaction closes |
| `revenue` | Computed post-sale |
| `profit` | Computed post-sale |
| `profit_margin_pct` | Computed post-sale |
| `selling_price` | Set AFTER markdown decision |
| `markdown_applied` | Operational outcome after assessment |
| `discount_pct` | Set after markdown decision |

> `spoilage_risk` = physics-based pre-computed score at receipt (keep — not leakage).  
> `quality_grade` = assessed by inspector at receipt (keep — not leakage).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, time, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from scipy.stats import randint

SEED = 42
np.random.seed(SEED)
print('Imports OK.')

---
## Step 1 — Load Data & Filter Pharmaceuticals

In [ ]:
df_raw = pd.read_csv('data/raw/7. perishable_goods_management.csv')
print(f'Raw shape          : {df_raw.shape}')

# Drop Pharmaceuticals — not food
df = df_raw[df_raw['category'] != 'Pharmaceuticals'].copy().reset_index(drop=True)
print(f'After filter shape : {df.shape}  ({len(df_raw)-len(df)} Pharmaceuticals rows removed)')

---
## Step 2 — Derive Target Label `risk_label` (HIGH / MODERATE / LOW)
Build the 3-class label **before** dropping any columns, because we need both `was_spoiled` and `days_until_expiry` to derive it.

In [ ]:
def derive_risk_label(row):
    """Derive HIGH/MODERATE/LOW from factual spoilage outcome + days remaining."""
    if row['was_spoiled'] == 1:
        return 'HIGH'                          # factually spoiled
    elif row['days_until_expiry'] <= 7:
        return 'HIGH'                          # < 1 week left, at high risk
    elif row['days_until_expiry'] <= 30:
        return 'MODERATE'                      # needs attention
    else:
        return 'LOW'                           # safe

df['risk_label'] = df.apply(derive_risk_label, axis=1)

print('Target label distribution:')
print(df['risk_label'].value_counts())
print()
print('Percentage breakdown:')
print((df['risk_label'].value_counts(normalize=True)*100).round(1).astype(str) + '%')

---
## Step 3 — Drop Leakage & Metadata Columns

In [ ]:
# Post-sale leakage columns
LEAKAGE_COLS = [
    'was_spoiled',       # used to build label — must NOT be a feature
    'units_wasted',      # counted after spoilage
    'waste_pct',         # derived from units_wasted
    'waste_cost',        # post-spoilage financial loss
    'units_sold',        # known after transaction closes
    'revenue',           # computed post-sale
    'profit',            # computed post-sale
    'profit_margin_pct', # computed post-sale
    'selling_price',     # set after markdown decision
    'markdown_applied',  # outcome after assessment
    'discount_pct',      # set after markdown
]

# ID/metadata not useful as features
META_COLS = [
    'record_id', 'product_id', 'product_name',
    'store_id', 'supplier_id', 'transaction_date', 'expiration_date',
]

df = df.drop(columns=LEAKAGE_COLS + META_COLS)
print(f'Columns remaining: {df.shape[1]}  (including risk_label)')
print(list(df.columns))

---
## Step 4 — Category Mapping & Feature/Target Split

In [ ]:
# Align category names with team canonical schema
df['category'] = df['category'].replace({'Produce': 'Fruits & Vegetables'})

TARGET = 'risk_label'
y = df[TARGET]
X = df.drop(columns=[TARGET])

print(f'X shape: {X.shape}  |  y shape: {y.shape}')
print(f'\nClass counts:\n{y.value_counts()}')

---
## Step 5 — EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
order = ['HIGH', 'MODERATE', 'LOW']
counts = y.value_counts().reindex(order)
colors = ['tomato', 'orange', 'steelblue']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[0].set_title('Class Distribution: risk_label', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(y)*100:.1f}%)', ha='center')

# Numeric feature correlation with HIGH class (1 vs rest)
y_binary = (y == 'HIGH').astype(int)
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
corr = X[numeric_cols].corrwith(y_binary).abs().sort_values(ascending=True)
corr.plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Feature Correlation with HIGH class', fontweight='bold')
axes[1].set_xlabel('|Pearson r|')

plt.tight_layout()
plt.savefig('notebooks/eda_advanced.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 6 — Feature Engineering & Encoding

In [ ]:
# Label-encode quality_grade (ordinal: A > B > C)
X = X.copy()
X['quality_grade'] = X['quality_grade'].map({'A': 2, 'B': 1, 'C': 0})

# One-hot encode nominal categoricals
X = pd.get_dummies(X, columns=['category', 'region'], drop_first=False, dtype=int)

FEATURE_COLS = list(X.columns)
print(f'Feature matrix: {X.shape}')
print(f'Features: {FEATURE_COLS}')

---
## Step 7 — Stratified Train / Test Split (80/20)
Stratify on `risk_label` to preserve HIGH/MODERATE/LOW proportions in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Train class dist: {y_train.value_counts(normalize=True).round(3).to_dict()}')
print(f'Test  class dist: {y_test.value_counts(normalize=True).round(3).to_dict()}')
print('✅ Stratification confirmed.')

---
## Step 8 — Feature Scaling (Post-Split ⚠️ Anti-Leakage)
Fit scaler **only on `X_train`** — never on the full dataset before splitting.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit + transform on TRAIN
X_test_sc  = scaler.transform(X_test)       # transform only on TEST

print(f'Scaling done. Train mean≈{X_train_sc.mean():.4f}, std≈{X_train_sc.std():.4f}')

---
## Step 9 — Baseline Model (Non-ML Rule)
Rule: Predict `HIGH` if `days_until_expiry ≤ 7`, `LOW` if `> 30`, otherwise `MODERATE`.

In [ ]:
dte_test = X_test['days_until_expiry'].values

def rule_predict(dte):
    if dte <= 7:   return 'HIGH'
    elif dte <= 30: return 'MODERATE'
    else:          return 'LOW'

y_baseline = pd.Series([rule_predict(d) for d in dte_test])

print('=== BASELINE (rule-based: DTE thresholds) ===')
print(f'Accuracy : {accuracy_score(y_test, y_baseline):.4f}')
print(f'F1 Macro : {f1_score(y_test, y_baseline, average="macro"):.4f}')
print(classification_report(y_test, y_baseline, target_names=['HIGH','LOW','MODERATE']))

---
## Step 10 — Model Training (3 Candidates)

In [ ]:
results = {}

def evaluate(name, model, Xtr, ytr, Xte, yte):
    t0 = time.time()
    model.fit(Xtr, ytr)
    elapsed = time.time() - t0
    y_pred = model.predict(Xte)
    acc = accuracy_score(yte, y_pred)
    f1  = f1_score(yte, y_pred, average='macro')
    results[name] = {'model': model, 'acc': acc, 'f1': f1, 'time': elapsed}
    print(f'\n--- {name} ---')
    print(f'  Accuracy: {acc:.4f} | F1 Macro: {f1:.4f} | Time: {elapsed:.1f}s')
    print(classification_report(yte, y_pred, target_names=['HIGH','LOW','MODERATE']))

evaluate('Logistic Regression',
         LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
         X_train_sc, y_train, X_test_sc, y_test)

evaluate('Random Forest',
         RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=SEED),
         X_train_sc, y_train, X_test_sc, y_test)

evaluate('Gradient Boosting',
         GradientBoostingClassifier(n_estimators=100, random_state=SEED),
         X_train_sc, y_train, X_test_sc, y_test)

---
## Step 11 — Hyperparameter Tuning (RandomizedSearchCV)
**Why RandomizedSearch?** 89K rows × Grid Search with many combos = hours. RandomizedSearch (`n_iter=30`) gives ~95% same quality in minutes using `n_jobs=-1` (all CPU cores).

In [ ]:
param_dist = {
    'n_estimators'      : randint(100, 400),
    'max_depth'         : [None, 10, 20, 30],
    'min_samples_split' : randint(2, 15),
    'min_samples_leaf'  : randint(1, 8),
    'max_features'      : ['sqrt', 'log2', 0.5],
}

search = RandomizedSearchCV(
    estimator   = RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=SEED),
    param_distributions = param_dist,
    n_iter      = 30,
    scoring     = 'f1_macro',
    cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs      = -1,
    verbose     = 1,
    random_state = SEED
)

print('Running RandomizedSearchCV...')
t0 = time.time()
search.fit(X_train_sc, y_train)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Best params : {search.best_params_}')
print(f'Best CV F1  : {search.best_score_:.4f}')

---
## Step 12 — Final Evaluation (Tuned Model)

### Error Risk per Class
| Error | Meaning | Business Impact |
|---|---|---|
| **Miss HIGH** (predict LOW/MOD, actual HIGH) | Spoilage not caught | ❌ Waste cost + food safety risk |
| **Miss LOW** (predict HIGH, actual LOW) | Unnecessary markdown | ⚠️ Lost margin |

> **Prioritise Recall for HIGH class.**

In [ ]:
best_model = search.best_estimator_
y_pred_best = best_model.predict(X_test_sc)

acc_best = accuracy_score(y_test, y_pred_best)
f1_best  = f1_score(y_test, y_pred_best, average='macro')

print('=== TUNED RANDOM FOREST — FINAL EVALUATION ===')
print(f'  Accuracy : {acc_best:.4f}')
print(f'  F1 Macro : {f1_best:.4f}')
print()
print(classification_report(y_test, y_pred_best, target_names=['HIGH','LOW','MODERATE']))

# Confusion Matrix
labels = ['HIGH', 'MODERATE', 'LOW']
cm = confusion_matrix(y_test, y_pred_best, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix — Tuned RF (Advanced Features)', fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('notebooks/confusion_advanced.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 13 — Feature Importance (Top 15)

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
top15 = importances.nlargest(15)

plt.figure(figsize=(8, 5))
top15.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances — Advanced RF', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('notebooks/feature_importance_advanced.png', dpi=120, bbox_inches='tight')
plt.show()
print(top15.round(4).to_string())

---
## Step 14 — Save Model
Saved as a dict package — same structure expected by `src/ml/spoilage.py`.

In [ ]:
os.makedirs('models', exist_ok=True)

model_package = {
    'model'    : best_model,
    'features' : FEATURE_COLS,
    'scaler'   : scaler,
    'version'  : 'advanced',
    'target'   : 'risk_label',                            # HIGH / MODERATE / LOW
    'classes'  : list(best_model.classes_),
    'notes'    : 'Perishable goods (food only). Advanced sensor+logistics features. Output: HIGH/MODERATE/LOW.'
}

joblib.dump(model_package, 'models/spoilage_classifier_advanced.pkl')
print('Saved: models/spoilage_classifier_advanced.pkl')
print(f'Classes  : {model_package["classes"]}')
print(f'Features : {len(FEATURE_COLS)} columns')

---
## Step 15 — Summary

| Model | Accuracy | F1 Macro | Notes |
|---|---|---|---|
| Baseline (DTE rule) | — | — | Non-ML reference |
| Logistic Regression | — | — | Linear |
| Random Forest (default) | — | — | |
| Gradient Boosting | — | — | |
| **Tuned Random Forest** ✅ | — | — | Saved |

> Fill from outputs above.

### Decision Checklist
| Aspek | Keputusan | Alasan | Alternatif |
|---|---|---|---|
| **Target** | `risk_label` HIGH/MOD/LOW | Match output contract `src/ml/spoilage.py` | `was_spoiled` binary → perlu konversi di backend |
| **Label derivation** | `was_spoiled` + DTE threshold | Kombinasi fakta aktual + konteks waktu tersisa | DTE threshold saja → 89% HIGH, model tidak belajar |
| **Imbalanced** | `class_weight='balanced'` + Stratified | 3 kelas tidak seimbang (67/23/9%) | SMOTE — lebih kompleks, butuh imblearn |
| **Scaling** | Setelah split, fit train only | Mencegah leakage statistik test set | Fit sebelum split → leakage, akurasi palsu |
| **Tuning** | RandomizedSearchCV (n_iter=30) | 89K baris — GridSearch terlalu lambat | GridSearch — exhaustive tapi berjam-jam |
| **CV** | StratifiedKFold (k=5) | Jaga proporsi 3 kelas di setiap fold | KFold biasa → fold bisa kehilangan kelas minority |
| **Metrik** | F1 Macro + Recall HIGH | Miss HIGH (spoilage tidak terdeteksi) paling mahal | Accuracy saja — menyesatkan pada data imbalanced |